# Binance ATR Backtest — Improved v2
### Config Section · Risk Management · Rich Summary · Analyst Review

**What is new vs v1:**
- All parameters in **one CONFIG cell** (Cell 2)
- Dollar-based **risk-per-trade sizing** (`RISK_PER_TRADE` % of equity)
- **Commission + slippage** modelled on every fill
- **Rich P&L summary** in both dollars and percent
- **Analyst review** explaining exactly why 68% win rate still loses money
- Synthetic BTC fallback (offline environments)

> Set `USE_SYNTHETIC = False` in Cell 2 to fetch real Binance data.


In [ ]:
# ================================================================
# CELL 1 — Imports
# ================================================================
import math, time, datetime as dt, warnings
warnings.filterwarnings('ignore')

import requests
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from typing import Optional, Tuple

pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 50)

print('Imports OK — pandas', pd.__version__, '| numpy', np.__version__)


---
## CONFIGURATION — Edit Everything Here

All strategy parameters, risk settings, and data options live in this single cell.  
You should not need to edit any other cell to tune the strategy.


In [ ]:
# ================================================================
# CELL 2 — MASTER CONFIG  (the only cell you need to edit)
# ================================================================

# -- DATA SOURCE -------------------------------------------------------
USE_SYNTHETIC     = True          # True = offline synthetic BTC; False = live Binance
SYMBOL            = 'BTCUSDT'
INTERVAL          = '1h'          # 1m 5m 15m 30m 1h 4h 1d
START             = dt.datetime(2023,  1,  1, tzinfo=dt.timezone.utc)
END               = dt.datetime(2024,  1,  1, tzinfo=dt.timezone.utc)

# -- RISK MANAGEMENT ---------------------------------------------------
INITIAL_CAPITAL   = 10_000.0      # Starting portfolio value ($)
RISK_PER_TRADE    = 0.01          # Fraction of equity risked per trade (0.01 = 1%)
MAX_POSITION      = 1.00          # Max fraction of equity in one position (1.0 = 100%)
COMMISSION_PCT    = 0.0004        # Per-side commission (Binance default = 0.04%)
SLIPPAGE_PCT      = 0.0002        # Per-fill slippage estimate

# -- INDICATOR SETTINGS ------------------------------------------------
ATR_LEN           = 14
SMA_FAST          = 20
SMA_SLOW          = 50

# -- STOP / TAKE PROFIT ------------------------------------------------
# KEY INSIGHT: for profitable trading, TP_MULT / STOP_MULT should be >= 2.0
# At your 68% win rate, break-even R:R = (1-0.68)/0.68 = 0.47x
# Anything above 0.47x R:R with 68% wins = profitable. Target >= 2.0x for buffer.
STOP_MULT         = 1.5           # Stop loss   = STOP_MULT * ATR
TP_MULT           = 3.5           # Take profit = TP_MULT   * ATR  ->  R:R = TP/STOP

# -- TRAILING STOP -----------------------------------------------------
USE_TRAILING      = True
TRAIL_MULT        = 1.8           # Trail gap = TRAIL_MULT * ATR (gap behind best price)
TRAIL_OFFSET_MULT = 1.0           # Trail activates after TRAIL_OFFSET_MULT * ATR of profit
#                                   0 = immediate trailing (cuts winners early — avoid)
#                                   1.0+ = let trade move 1R before trailing kicks in

# -- DISPLAY -----------------------------------------------------------
DARK_MODE         = True

print('Config loaded:')
print(f'  Symbol : {SYMBOL} {INTERVAL}')
print(f'  Capital: ${INITIAL_CAPITAL:,.0f}  |  Risk/trade: {RISK_PER_TRADE*100:.1f}%')
print(f'  Stop   : {STOP_MULT}x ATR  |  TP: {TP_MULT}x ATR  |  R:R = {TP_MULT/STOP_MULT:.2f}x')
print(f'  Commission: {COMMISSION_PCT*100:.3f}%/side  |  Trailing: {"ON" if USE_TRAILING else "OFF"}')
print(f'  Break-even win rate at {TP_MULT/STOP_MULT:.2f}x R:R = {1/(1+TP_MULT/STOP_MULT)*100:.1f}%')


---
## Data Fetching


In [ ]:
# ================================================================
# CELL 3 — DATA  (Binance fetch or synthetic fallback)
# ================================================================

BINANCE_BASE = 'https://api.binance.com'

def to_millis(ts):
    if ts.tzinfo is None:
        ts = ts.replace(tzinfo=dt.timezone.utc)
    return int(ts.timestamp() * 1000)

def fetch_klines(symbol, interval, start_time, end_time, limit=1000, pause=0.3):
    start_ms, end_ms = to_millis(start_time), to_millis(end_time)
    url, all_rows, cur = f'{BINANCE_BASE}/api/v3/klines', [], start_ms
    while cur < end_ms:
        params = dict(symbol=symbol.upper(), interval=interval,
                      startTime=cur, endTime=end_ms, limit=limit)
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        batch = resp.json()
        if not batch: break
        all_rows.extend(batch)
        if len(batch) < limit: break
        cur = batch[-1][0] + 1
        time.sleep(pause)
    if not all_rows:
        return pd.DataFrame()
    cols = ['open_time','open','high','low','close','volume','close_time',
            'qav','num_trades','tbbav','tbqav','ignore']
    df = pd.DataFrame(all_rows, columns=cols)
    for c in ['open','high','low','close','volume']:
        df[c] = pd.to_numeric(df[c])
    df['open_time'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)
    return df[['open_time','open','high','low','close','volume']].set_index('open_time').sort_index()

def generate_synthetic_btc(start='2023-01-01', n_hours=8760, seed=42):
    '''Regime-switching GBM synthetic BTC OHLCV.'''
    rng = np.random.default_rng(seed)
    price, prices = 16_500.0, [16_500.0]
    i, regs, lens = 0, [], []
    while i < n_hours:
        r = rng.choice(['bull','bear','chop'], p=[0.45, 0.25, 0.30])
        l = int(rng.integers(100, 400))
        lens.append(min(l, n_hours - i)); regs.append(r); i += l
    params = {'bull':(0.00055,0.016), 'bear':(-0.00035,0.018), 'chop':(0.00003,0.010)}
    for reg, le in zip(regs, lens):
        mu, sig = params[reg]
        for _ in range(le):
            price *= np.exp(rng.normal(mu, sig))
            prices.append(price)
    prices = np.array(prices[:n_hours+1])
    closes, opens = prices[1:], prices[:-1]
    noise = rng.uniform(0.003, 0.018, n_hours)
    highs = np.maximum(opens, closes) * (1 + noise * rng.uniform(0.4, 1.0, n_hours))
    lows  = np.minimum(opens, closes) * (1 - noise * rng.uniform(0.4, 1.0, n_hours))
    vol   = 500 * rng.lognormal(0, 0.65, n_hours) * (1 + 2*np.abs(closes/opens - 1)*80)
    idx   = pd.date_range(start, periods=n_hours, freq='1h', tz='UTC')
    df    = pd.DataFrame(dict(open=opens,high=highs,low=lows,close=closes,volume=vol), index=idx)
    df.index.name = 'open_time'
    return df

# -- Load data ---------------------------------------------------------
if USE_SYNTHETIC:
    n_hours = int((END - START).total_seconds() / 3600)
    df_raw  = generate_synthetic_btc(str(START.date()), n_hours)
    print(f'Synthetic data: {len(df_raw):,} bars | {df_raw.index[0].date()} to {df_raw.index[-1].date()}')
else:
    print(f'Fetching {SYMBOL} {INTERVAL} from Binance...')
    df_raw = fetch_klines(SYMBOL, INTERVAL, START, END)
    if df_raw.empty:
        raise SystemExit('No data returned — check symbol/interval/date range.')
    print(f'Fetched {len(df_raw):,} bars | {df_raw.index[0].date()} to {df_raw.index[-1].date()}')

print(f'Price range: ${df_raw.close.min():,.0f} to ${df_raw.close.max():,.0f}')
df_raw.tail(3)


---
## Indicators & Signals

Replace `generate_signals()` with your own entry logic.  
Everything else (engine, sizing, summary) stays unchanged.


In [ ]:
# ================================================================
# CELL 4 — INDICATORS & SIGNALS
# ================================================================

def compute_atr(df, length=14):
    h, l, c = df['high'], df['low'], df['close']
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    return tr.rolling(length, min_periods=length).mean()

def compute_sma(s, length):
    return s.rolling(length, min_periods=length).mean()

def compute_ema(s, length):
    return s.ewm(span=length, adjust=False).mean()

def compute_rsi(s, period=14):
    d = s.diff()
    g = d.clip(lower=0).ewm(alpha=1/period, adjust=False).mean()
    l = (-d.clip(upper=0)).ewm(alpha=1/period, adjust=False).mean()
    return 100 - 100 / (1 + g / l.replace(0, np.nan))

def generate_signals(df):
    '''
    Default: SMA crossover (same as original notebook).

    HOW TO PLUG IN YOUR OWN SIGNALS:
    Return (long_signal, short_signal) as boolean pd.Series aligned to df.index.

    Example — EMA crossover + RSI + trend filter:
        ema_fast = compute_ema(df["close"], SMA_FAST)
        ema_slow = compute_ema(df["close"], SMA_SLOW)
        sma200   = compute_sma(df["close"], 200)
        rsi      = compute_rsi(df["close"], 14)
        long  = (ema_fast > ema_slow) & (ema_fast.shift(1) <= ema_slow.shift(1))\
                & (df["close"] > sma200) & (rsi > 50) & (rsi < 70)
        short = (ema_fast < ema_slow) & (ema_fast.shift(1) >= ema_slow.shift(1))\
                & (df["close"] < sma200) & (rsi < 50) & (rsi > 30)
        return long.fillna(False), short.fillna(False)
    '''
    sf = compute_sma(df['close'], SMA_FAST)
    ss = compute_sma(df['close'], SMA_SLOW)
    long_sig  = (sf > ss) & (sf.shift(1) <= ss.shift(1))
    short_sig = (sf < ss) & (sf.shift(1) >= ss.shift(1))
    return long_sig.fillna(False), short_sig.fillna(False)


# -- Build indicator DataFrame -----------------------------------------
df = df_raw.copy()
df['atr']      = compute_atr(df, ATR_LEN)
df['sma_fast'] = compute_sma(df['close'], SMA_FAST)
df['sma_slow'] = compute_sma(df['close'], SMA_SLOW)
df['rsi']      = compute_rsi(df['close'], 14)

long_sig, short_sig = generate_signals(df)

df['stop_dist']    = STOP_MULT         * df['atr']
df['tp_dist']      = TP_MULT           * df['atr']
df['trail_points'] = TRAIL_MULT        * df['atr'] if USE_TRAILING else np.nan
df['trail_offset'] = TRAIL_OFFSET_MULT * df['atr'] if USE_TRAILING else np.nan

print(f'Indicators computed | Long signals: {long_sig.sum()} | Short signals: {short_sig.sum()}')
print(f'Avg ATR: ${df["atr"].dropna().mean():,.0f} ({(df["atr"]/df["close"]*100).dropna().mean():.2f}% of price)')
print(f'Avg stop: ${df["stop_dist"].dropna().mean():,.0f} | Avg TP: ${df["tp_dist"].dropna().mean():,.0f} | R:R = {TP_MULT/STOP_MULT:.2f}x')


---
## Backtest Engine

Improvements vs v1:
- **Risk-based sizing** — each trade risks exactly `RISK_PER_TRADE x equity`
- **Commission + slippage** deducted on every fill
- **Delayed trailing** — trailing activates only after `TRAIL_OFFSET_MULT x ATR` of profit
- **Dollar P&L** tracked per trade


In [ ]:
# ================================================================
# CELL 5 — BACKTEST ENGINE
# ================================================================

equity       = INITIAL_CAPITAL
pos          = None       # active position dict, None when flat
pending      = None       # signal queued for next bar open
trades_raw   = []
equity_curve = [equity]   # one equity snapshot per bar

for i in range(1, len(df)):
    row  = df.iloc[i]
    prev = df.iloc[i - 1]
    bh, bl, bo = row.high, row.low, row.open

    # -- 1. Execute pending entry at this bar open -------------------------
    if pending is not None and pos is None:
        side, sig_t, sl_d, tp_d, tp_pts, tp_off, qty = pending
        fill   = bo * (1 + SLIPPAGE_PCT) if side == 'long' else bo * (1 - SLIPPAGE_PCT)
        equity -= equity * qty * COMMISSION_PCT   # entry commission
        sl = fill - sl_d if side == 'long' else fill + sl_d
        tp = fill + tp_d if side == 'long' else fill - tp_d
        pos = dict(
            side=side, entry=fill, sl=sl, sl_orig=sl, tp=tp,
            trail_pts=tp_pts, trail_off=tp_off, qty=qty,
            eq_at_entry=equity, highest=fill, lowest=fill,
            trail_active=False, entry_t=sig_t, entry_bar=i,
        )
        pending = None

    # -- 2. Manage open position ------------------------------------------
    if pos is not None:
        d = pos
        exit_price  = None
        exit_reason = None

        if d['side'] == 'long':
            d['highest'] = max(d['highest'], bh)
            # activate trailing only after TRAIL_OFFSET_MULT * ATR of profit
            if d['trail_pts'] and not d['trail_active']:
                if (d['highest'] - d['entry']) >= d['trail_off']:
                    d['trail_active'] = True
            # ratchet trailing stop upward
            if d['trail_pts'] and d['trail_active']:
                d['sl'] = max(d['sl'], d['highest'] - d['trail_pts'])
            # check exits (stop has priority on same bar)
            if bl <= d['sl']:
                exit_price  = d['sl'] * (1 - SLIPPAGE_PCT)
                exit_reason = 'trail_stop' if d['trail_active'] else 'stop'
            elif bh >= d['tp']:
                exit_price  = d['tp'] * (1 - SLIPPAGE_PCT)
                exit_reason = 'tp'

        else:  # short
            d['lowest'] = min(d['lowest'], bl)
            if d['trail_pts'] and not d['trail_active']:
                if (d['entry'] - d['lowest']) >= d['trail_off']:
                    d['trail_active'] = True
            if d['trail_pts'] and d['trail_active']:
                d['sl'] = min(d['sl'], d['lowest'] + d['trail_pts'])
            if bh >= d['sl']:
                exit_price  = d['sl'] * (1 + SLIPPAGE_PCT)
                exit_reason = 'trail_stop' if d['trail_active'] else 'stop'
            elif bl <= d['tp']:
                exit_price  = d['tp'] * (1 + SLIPPAGE_PCT)
                exit_reason = 'tp'

        if exit_price is not None:
            pnl_pct    = ((exit_price - d['entry']) / d['entry']
                          if d['side'] == 'long'
                          else (d['entry'] - exit_price) / d['entry'])
            commission = equity * d['qty'] * COMMISSION_PCT
            pnl_dollar = d['eq_at_entry'] * d['qty'] * pnl_pct - commission
            equity    += pnl_dollar
            trades_raw.append(dict(
                side        = d['side'],
                entry_time  = d['entry_t'],
                exit_time   = row.name,
                entry_price = d['entry'],
                exit_price  = exit_price,
                exit_reason = exit_reason,
                bars_held   = i - d['entry_bar'],
                pnl_pts     = exit_price - d['entry'] if d['side']=='long' else d['entry']-exit_price,
                pnl_pct     = pnl_pct * 100,
                pnl_dollar  = pnl_dollar,
                equity      = equity,
            ))
            pos = None

    # -- 3. Check for new signal at prev bar close -------------------------
    if pos is None:
        atr_val = float(prev['atr'])
        if not math.isnan(atr_val) and atr_val > 0:
            sl_d   = float(prev['stop_dist'])
            tp_d   = float(prev['tp_dist'])
            tp_pts = float(prev['trail_points']) if USE_TRAILING and not math.isnan(float(prev['trail_points'])) else None
            tp_off = float(prev['trail_offset']) if USE_TRAILING and not math.isnan(float(prev['trail_offset'])) else None
            # risk-based sizing: loss on SL hit = RISK_PER_TRADE * equity
            stop_pct = sl_d / float(prev['close'])
            qty      = min(RISK_PER_TRADE / stop_pct, MAX_POSITION) if stop_pct > 0 else 0
            if long_sig.iloc[i - 1]:
                pending = ('long',  prev.name, sl_d, tp_d, tp_pts, tp_off, qty)
            elif short_sig.iloc[i - 1]:
                pending = ('short', prev.name, sl_d, tp_d, tp_pts, tp_off, qty)

    equity_curve.append(equity)

# -- Build trades DataFrame --------------------------------------------
trades = pd.DataFrame(trades_raw)
print(f'Backtest complete: {len(trades)} closed trades')
if not trades.empty:
    wins = trades[trades.pnl_dollar > 0]
    print(f'Win rate: {len(wins)/len(trades)*100:.1f}% | Final equity: ${equity:,.2f} | Net: ${equity-INITIAL_CAPITAL:+,.2f} ({(equity/INITIAL_CAPITAL-1)*100:+.1f}%)')


---
## Performance Summary


In [ ]:
# ================================================================
# CELL 6 — PERFORMANCE SUMMARY
# ================================================================

def compute_summary(trades, equity_curve, initial_capital):
    if trades.empty:
        return {}
    wins   = trades[trades.pnl_dollar > 0]
    losses = trades[trades.pnl_dollar <= 0]
    n      = len(trades)
    gp     = wins.pnl_dollar.sum()
    gl     = losses.pnl_dollar.sum()
    pf     = (gp / abs(gl)) if gl < 0 else float('inf')
    final  = equity_curve[-1]
    net    = final - initial_capital
    net_pct= (final / initial_capital - 1) * 100
    ec     = pd.Series(equity_curve)
    dd     = ((ec - ec.cummax()) / ec.cummax() * 100).min()
    rets   = ec.pct_change().dropna()
    sharpe = (rets.mean() / rets.std()) * (24 * 365) ** 0.5 if rets.std() > 0 else 0.0
    avg_w  = wins.pnl_dollar.mean()   if len(wins)   else 0.0
    avg_l  = losses.pnl_dollar.mean() if len(losses) else 0.0
    wr     = len(wins) / n
    ev     = wr * avg_w + (1 - wr) * avg_l
    rr_r   = abs(avg_w / avg_l) if avg_l != 0 else float('inf')
    by_r   = trades.groupby('exit_reason')['pnl_dollar'].count().to_dict()
    return dict(
        n=n, wins=len(wins), losses=len(losses), win_rate=wr*100,
        pf=pf, initial=initial_capital, final=final,
        net_dollar=net, net_pct=net_pct, gp=gp, gl=gl,
        avg_win=avg_w, avg_loss=avg_l, rr=rr_r, rr_target=TP_MULT/STOP_MULT,
        ev=ev, max_dd=dd, sharpe=sharpe,
        avg_bars=trades.bars_held.mean(), by_reason=by_r)


def print_summary(s):
    sep  = '=' * 62
    sep2 = '-' * 62
    pnl_sign = '+' if s['net_dollar'] >= 0 else ''
    wr_be    = 1 / (1 + s['rr']) * 100 if not math.isinf(s['rr']) else 0.0
    wr_ok    = s['win_rate'] >= wr_be
    ev_ok    = s['ev'] > 0
    print(sep)
    print('  BACKTEST PERFORMANCE SUMMARY')
    print(f"  {SYMBOL} {INTERVAL}  |  Capital: ${s['initial']:>10,.2f}")
    print(sep)
    print('  OVERVIEW')
    print(sep2)
    print(f"  Total Trades        : {s['n']}")
    print(f"  Winners / Losers    : {s['wins']} / {s['losses']}")
    print(f"  Win Rate            : {s['win_rate']:.1f}%")
    print(f"  Avg Bars Held       : {s['avg_bars']:.1f}")
    print()
    print('  P&L')
    print(sep2)
    print(f"  Initial Capital     : ${s['initial']:>12,.2f}")
    print(f"  Final Equity        : ${s['final']:>12,.2f}")
    print(f"  Net P&L ($)         : {pnl_sign}${s['net_dollar']:>11,.2f}")
    print(f"  Net P&L (%)         : {pnl_sign}{s['net_pct']:.2f}%")
    print(f"  Gross Profit ($)    : +${s['gp']:>10,.2f}")
    print(f"  Gross Loss ($)      :  ${s['gl']:>10,.2f}")
    print()
    print('  RISK & EFFICIENCY')
    print(sep2)
    print(f"  Profit Factor       : {s['pf']:.3f}")
    print(f"  Expected Value/Trade: ${s['ev']:>+,.2f}")
    print(f"  Avg Win ($)         : +${s['avg_win']:>9,.2f}")
    print(f"  Avg Loss ($)        :  ${s['avg_loss']:>9,.2f}")
    print(f"  Realised R:R        : {s['rr']:.2f}x  (target: {s['rr_target']:.2f}x)")
    print(f"  Max Drawdown        : {s['max_dd']:.1f}%")
    print(f"  Sharpe Ratio        : {s['sharpe']:.2f}")
    print()
    print('  EXIT BREAKDOWN')
    print(sep2)
    for reason, count in s.get('by_reason', {}).items():
        pct = count / s['n'] * 100
        print(f'  {reason:<18} : {count:>4} trades  ({pct:.1f}%)')
    print()
    print('  BREAK-EVEN ANALYSIS')
    print(sep2)
    print(f"  Realised R:R        : {s['rr']:.2f}x")
    print(f"  Break-even Win Rate : {wr_be:.1f}%")
    print(f"  Your Win Rate       : {s['win_rate']:.1f}%  ({'ABOVE break-even' if wr_ok else 'BELOW break-even'})")
    print(f"  Expected Value      : ${s['ev']:+,.2f}/trade  ({'edge exists' if ev_ok else 'NO EDGE — losing system'})")
    print(sep)


summary = compute_summary(trades, equity_curve, INITIAL_CAPITAL)
if summary:
    print_summary(summary)
else:
    print('No closed trades — run Cell 5 first.')


---
## Charts


In [ ]:
# ================================================================
# CELL 7 — CHARTS
# ================================================================

if trades.empty:
    print('No trades to plot.')
else:
    BG  = '#0d0f14' if DARK_MODE else '#f8f9fa'
    FG  = '#c8d0e0' if DARK_MODE else '#1a1a2e'
    DIM = '#8892a4' if DARK_MODE else '#6c757d'
    GRD = '#1a1f2e' if DARK_MODE else '#dee2e6'
    GRN, RED, YLW, BLU = '#00e5a0', '#ff4466', '#f0c040', '#4a9eff'

    fig = plt.figure(figsize=(18, 22), facecolor=BG)
    gs  = gridspec.GridSpec(4, 2, figure=fig,
                            height_ratios=[2.2, 1.2, 1.2, 1.2],
                            hspace=0.45, wspace=0.28)
    ax_p  = fig.add_subplot(gs[0, :])
    ax_eq = fig.add_subplot(gs[1, :])
    ax_dd = fig.add_subplot(gs[2, :])
    ax_pl = fig.add_subplot(gs[3, 0])
    ax_bh = fig.add_subplot(gs[3, 1])

    for ax in [ax_p, ax_eq, ax_dd, ax_pl, ax_bh]:
        ax.set_facecolor(BG)
        ax.tick_params(colors=DIM, labelsize=8.5)
        for sp in ax.spines.values(): sp.set_edgecolor(GRD)

    def ttl(ax, t): ax.set_title(t, color=FG, fontsize=10, pad=6,
                                  fontfamily='monospace', fontweight='bold')

    # Price chart
    s = df.iloc[-min(2000, len(df)):]
    ax_p.plot(s.index, s.close,    color=FG,  lw=0.65, alpha=0.8,  label='Close')
    ax_p.plot(s.index, s.sma_fast, color=YLW, lw=1.0,  alpha=0.85, label=f'SMA{SMA_FAST}')
    ax_p.plot(s.index, s.sma_slow, color=BLU, lw=1.0,  alpha=0.85, label=f'SMA{SMA_SLOW}')
    for _, t in trades.iterrows():
        try:
            c  = GRN if t.side == 'long' else RED
            mk = '^' if t.side == 'long' else 'v'
            tc = YLW if t.exit_reason == 'tp' else '#555'
            ax_p.scatter(t.entry_time, t.entry_price, color=c,  marker=mk,  s=55, zorder=5)
            ax_p.scatter(t.exit_time,  t.exit_price,  color=tc, marker='x', s=45, zorder=5)
        except Exception:
            pass
    ax_p.legend(loc='upper left', fontsize=8, facecolor='#111520' if DARK_MODE else '#fff',
                edgecolor=GRD, labelcolor=FG)
    ttl(ax_p, 'PRICE  |  SMA signals  |  triangle=entry  x=exit  (yellow=TP  grey=SL)')
    ax_p.set_ylabel('Price (USDT)', color=DIM, fontsize=9)
    ax_p.grid(True, color=GRD, lw=0.4, alpha=0.5)

    # Equity curve
    ec_s = pd.Series(equity_curve, index=df.index[:len(equity_curve)])
    ax_eq.plot(ec_s.index, ec_s.values, color=GRN, lw=1.5)
    ax_eq.axhline(INITIAL_CAPITAL, color=DIM, lw=0.8, ls='--', alpha=0.5)
    ax_eq.fill_between(ec_s.index, INITIAL_CAPITAL, ec_s.values,
                       where=ec_s.values >= INITIAL_CAPITAL, alpha=0.15, color=GRN)
    ax_eq.fill_between(ec_s.index, INITIAL_CAPITAL, ec_s.values,
                       where=ec_s.values <  INITIAL_CAPITAL, alpha=0.15, color=RED)
    ttl(ax_eq, 'EQUITY CURVE  ($)')
    ax_eq.set_ylabel('Portfolio ($)', color=DIM, fontsize=9)
    ax_eq.grid(True, color=GRD, lw=0.4, alpha=0.5)

    # Drawdown
    roll_max  = ec_s.cummax()
    dd_series = (ec_s - roll_max) / roll_max * 100
    ax_dd.fill_between(dd_series.index, dd_series.values, 0, color=RED, alpha=0.5)
    ax_dd.plot(dd_series.index, dd_series.values, color=RED, lw=0.9)
    ttl(ax_dd, 'DRAWDOWN  (%)')
    ax_dd.set_ylabel('DD (%)', color=DIM, fontsize=9)
    ax_dd.grid(True, color=GRD, lw=0.4, alpha=0.5)

    # Per-trade P&L ($)
    clrs = [GRN if v > 0 else RED for v in trades.pnl_dollar]
    ax_pl.bar(range(len(trades)), trades.pnl_dollar, color=clrs, alpha=0.85, width=0.8)
    ax_pl.axhline(0, color=DIM, lw=0.8, ls='--')
    ttl(ax_pl, 'TRADE P&L  ($)')
    ax_pl.set_xlabel('Trade #', color=DIM, fontsize=8)
    ax_pl.set_ylabel('P&L ($)', color=DIM, fontsize=9)
    ax_pl.grid(True, color=GRD, lw=0.4, alpha=0.5)

    # Bars held histogram
    ax_bh.hist(trades.bars_held, bins=20, color=BLU, alpha=0.78, edgecolor=GRD)
    ttl(ax_bh, 'TRADE DURATION  (bars)')
    ax_bh.set_xlabel('Bars held', color=DIM, fontsize=8)
    ax_bh.set_ylabel('Count', color=DIM, fontsize=9)
    ax_bh.grid(True, color=GRD, lw=0.4, alpha=0.5)

    fig.suptitle(f'{SYMBOL}  {INTERVAL}  |  Stop {STOP_MULT}x  TP {TP_MULT}x  R:R {TP_MULT/STOP_MULT:.1f}x',
                 color=FG, fontsize=13, fontfamily='monospace', fontweight='bold', y=0.999)
    plt.savefig(f'backtest_{SYMBOL}_{INTERVAL}.png', dpi=130, bbox_inches='tight', facecolor=BG)
    plt.show()
    print(f'Chart saved: backtest_{SYMBOL}_{INTERVAL}.png')


---
## Analyst Review — Why 68% Win Rate Still Loses Money

This is the most important section. The mathematics here explains exactly why a high win rate can still produce negative P&L, and what to do about it.


In [ ]:
# ================================================================
# CELL 8 — ANALYST REVIEW & DIAGNOSIS
# ================================================================

if not trades.empty and summary:
    wr  = summary['win_rate'] / 100
    rr  = summary['rr']
    be  = 1 / (1 + rr) * 100 if not math.isinf(rr) else 0.0

    print('THE CORE MATHS')
    print('-' * 62)
    print('Profitability requires BOTH win rate AND R:R to be aligned.')
    print()
    print('  EV = (WinRate x AvgWin) + (LossRate x AvgLoss)')
    print()
    print('  For EV > 0:  WinRate > 1 / (1 + R:R)')
    print()
    print('  R:R     Break-even Win Rate')
    print('  0.5x    67%')
    print('  1.0x    50%')
    print('  1.5x    40%')
    print('  2.0x    33%')
    print('  2.5x    29%')
    print('  3.0x    25%  (1 win covers 3 losses)')
    print()
    print('YOUR RESULTS')
    print('-' * 62)
    print(f"  Win Rate         : {summary['win_rate']:.1f}%")
    print(f"  Realised R:R     : {rr:.2f}x")
    print(f"  Break-even at    : {be:.1f}% win rate")
    print(f"  Avg Win          : ${summary['avg_win']:+,.2f}")
    print(f"  Avg Loss         : ${summary['avg_loss']:+,.2f}")
    print(f"  Expected Value   : ${summary['ev']:+,.2f}/trade  (" +
          ('edge exists' if summary['ev'] > 0 else 'NEGATIVE — losing system') + ')')
    print()
    print('THE 68% WIN RATE TRAP')
    print('-' * 62)
    print('68% sounds excellent. But if avg_loss > avg_win (R:R < 0.47x),')
    print('you need >68% wins just to break even.')
    print()
    print('Common cause: TP is too close, SL is too wide.')
    print('  TP = 0.5x ATR,  SL = 2.0x ATR  ->  R:R = 0.25x')
    print('  Break-even requires 80% wins. 68% loses money.')
    print()
    print('FIVE FIXES — IN ORDER OF IMPACT')
    print('-' * 62)
    print('1. FIX YOUR R:R FIRST')
    print('   Set TP_MULT / STOP_MULT >= 2.0')
    print('   Your 68% win rate + 2.0x R:R = exceptional system.')
    print()
    print('2. DELAY THE TRAILING STOP')
    print('   Set TRAIL_OFFSET_MULT = 1.0 (activates after +1R of profit)')
    print('   Immediate trailing crushes avg_win. Let winners breathe first.')
    print()
    print('3. ADD A PARTIAL TP AT 1R')
    print('   Close 50% at +1x stop distance, move stop to break-even.')
    print('   Turns marginal trades from losses into flat or small wins.')
    print()
    print('4. FILTER OUT CHOPPY MARKETS')
    print('   Only trade when close > SMA(200)  (trend filter)')
    print('   Only trade when ATR > SMA(ATR, 30)  (volatility filter)')
    print('   Eliminates 30-40% of losing trades with minimal signal loss.')
    print()
    print('5. REDUCE TRADE FREQUENCY')
    print('   Fewer, higher-quality signals beat frequent noisy ones.')
    print('   Every filter that removes bad trades improves expected value.')
    print()
    print('WHAT YOUR 68% WIN RATE IS WORTH AT DIFFERENT R:R')
    print('-' * 62)
    test_wr = 0.68
    for test_rr in [0.5, 0.8, 1.0, 1.5, 2.0, 2.5, 3.0]:
        ev_n = test_wr * test_rr + (1 - test_wr) * (-1.0)
        be_n = 1 / (1 + test_rr) * 100
        tag  = 'PROFITABLE' if ev_n > 0 else 'losing    '
        print(f'  R:R {test_rr:.1f}x  |  BE: {be_n:.0f}%  |  EV: {ev_n:+.3f}  ({tag})')
    be_exact = (1 - 0.68) / 0.68
    print(f'  Your break-even R:R at 68% wins = {be_exact:.2f}x')
    print(f'  Any R:R above {be_exact:.2f}x = profitable with your signal quality.')
    print()
    print('BOTTOM LINE')
    print('-' * 62)
    print('Your signal quality is good — 68% is genuinely high.')
    print('The problem is entirely in exit management (TP too close / SL too wide).')
    print('Widen TP_MULT, keep or tighten STOP_MULT, and you likely')
    print('already have a profitable system.')
else:
    print('Run Cells 2-5 first.')


---
## Export Trades


In [ ]:
# ================================================================
# CELL 9 — EXPORT
# ================================================================
if not trades.empty:
    fname = f'trades_{SYMBOL}_{INTERVAL}.csv'
    trades.to_csv(fname, index=False)
    print(f'Saved {len(trades)} trades to {fname}')
    trades.tail(10)
else:
    print('No trades to export.')
